# NB-003_regression_gates.ipynb — Mod 03 Regression Gates (notebook-first prototype, D2)Mod 03 — Regression Gates: notebook-first prototype (D2), promotes into `src/llmops/eval/`.

**Links** — companion LLD: `doc/design/03_lld_tests.md`; task 03 contract: `doc/task/03_regression_gates.md`; step4 parity reference: `data/docs/s4_qa_deep_dives.md:487,500-502` (regression & quality gates row + Interview Q2).

**Scope banner** — offline-deterministic only; live/judge path is OUT (D5); Mod 4 owns latency computation.

**Offline seam note** — this notebook must never read `.env` / `LLM_*` / `GROQ_*` — offline seam (D5/D6, T-03-11).

## Section 1 — `metric_registry` prototypingMetric schema is DATA, not code — `Metric` dataclass, `MetricKind`/`MetricDirection` TypeAliases.

**Tolerance policy** — judge-default ±0.03 absolute vs per-source 1/(n+1) single-flip floor vs latency ±0.20 relative (D10/D19).

**Reconciliation note** — 139 (retriever) + 8 (misroute) + 107 (correctness) + 59 (ungated categories) = 313 — verified against golden files.

Inline sanity-check narrative → promotes to T-03-1.

## Section 2 — `run_suite` prototypingOffline deterministic evaluator, pure function of committed goldens (+ corpus ONLY for `corpus_sha256` and `golden_rules` L1 checks).

**Explicit non-goals** — no network, no env reads, no `config/judge.py` import (D5/D6, offline seam).

**Normalization contract** — NFC, line-ending normalize, trim, case-preserved substring match — owned by `t01_verify` helper, imported (not subprocess).

**Determinism narrative** — canonical-bytes subset (schema_version + sorted metrics, `generated_utc` excluded) → promotes to T-03-2.

**Empty `must_contain` trap** — runtime guard → `EvaluationInputError` (H8); empty goldens dir → exit 3 (T-03-2a).

## Section 3 — Manual exploration / dry runsRun against the committed goldens, eyeball the report before wiring tests.

**Expected-value narrative** — every gate row should read 1.0 on committed goldens (T-03-4).

Note: this section is throwaway exploration, not part of the promoted module.

## Section 4 — `snapshot` prototypingBaseline serialization contract — committed, atomic write, provenance fields.

**Canonical pointer (`active.json`)** — schema `{schema_version, baseline_id, path}`, path-safety rules (basename only, no traversal), resolved before compare (T-03-5b).

**Baseline-change policy** — baseline + pointer change ship in the same commit (registered change, D19-analogous).

## Section 5 — Build + commit the first baselineOne-time act: create `eval/baselines/<id>.json` + `active.json`.

Reminder: baseline commit is reviewed, not silent (D19-analogous registered change).

## Section 6 — `compare` prototypingVerdict engine — 11-step precedence, structural checks BEFORE value comparison.

**Boundary semantics** — inclusive band, computed bound (e.g. `n/(n+1)` floor), no rounding before verdicting (compare INVARIANT).

## Section 7 — CLI (`main`) prototyping`--baseline` vs `--active`, `--candidate`, exit code 0–4 contract.

**Error taxonomy recap (0/1/2/3/4)** — 0=PASS, 1=FAIL (regression incl. unknown candidate id, missing candidate gate, coverage regression), 2=REVIEW, 3=EvaluationInputError (malformed report / duplicate ids / empty goldens), 4=ConfigurationError (baseline missing / pointer violations / missing baseline gate / zero-baseline relative). argparse usage errors → 3 via custom `error()` override (native exit 2 reserved for REVIEW, D41).

## Section 8 — Interactive verification against the test matrixRepresentative cases from the LLD test matrix as notebook demos (NOT a replacement for `tests/test_gates.py`).

Note: each demo cell's expected exit code is stated inline before running.

## Section 9 — Offline-seam self-checkProve the module never imports live/judge code or touches env.

Note: full enforcement (env spy, read-only/socket backstop, deny-set subset) lives in `tests/test_gates.py` (T-03-11/11b/11c/11d), not the notebook.

## Section 10 — Promotion checklistMove finalized cells into `src/llmops/eval/{metric_registry,run_suite,snapshot,compare}.py` + `__init__.py` facade.

Checklist: registry validated; run_suite deterministic; snapshot round-trips; compare precedence matches 11-step spec; CLI exit codes verified.

Reminder: promoted code must additionally pass `tests/test_gates.py`, `ruff`, `mypy` (T-03-12) and the CI workflow contract (`llm_eval_gate.yml`) before merge.

## Section 11 — Verify block (mirrors task 03)Reproduce the LLD's Verify commands post-promotion, from shell:

- `uv run python -m llmops.eval.compare --baseline eval/baselines/<id>.json --candidate <report>` + `echo $?` mapping (0..4)- `uv run pytest tests/test_gates.py -q`- `uv run ruff check .`- `uv run mypy src/`

Closing note: this notebook is the scratchpad; `src/llmops/eval/` + `tests/test_gates.py` are the artifacts of record.